In [1]:
import os
import re
from dotenv import load_dotenv
from langchain.embeddings import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI

In [2]:
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
#embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

/var/folders/m8/lr5c6v793qbcszm54dsf6y440000gn/T/ipykernel_15385/974406059.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)


In [4]:
# langgraph_agent.py

from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
import operator

# ---------------------------------------------------
# 1️⃣ Define Agent State
# ---------------------------------------------------

class AgentState(TypedDict):
    input: str
    output: str
    path: str


# ---------------------------------------------------
# 2️⃣ Define Tools
# ---------------------------------------------------

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"


wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())


# ---------------------------------------------------
# 3️⃣ Initialize LLM
# ---------------------------------------------------

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# ---------------------------------------------------
# 4️⃣ Routing Logic
# ---------------------------------------------------

def router(state: AgentState):
    query = state["input"].lower()

    if any(op in query for op in ["+", "-", "*", "/"]) or "calculate" in query:
        return "calculator_node"
    elif "who" in query or "when" in query or "search" in query:
        return "wikipedia_node"
    else:
        return "llm_node"


# ---------------------------------------------------
# 5️⃣ Tool Nodes
# ---------------------------------------------------

def calculator_node(state: AgentState):
    expression = state["input"].replace("calculate", "")
    result = calculator.invoke(expression)
    return {"output": result, "path": "calculator"}


def wikipedia_node(state: AgentState):
    result = wiki.run(state["input"])
    return {"output": result, "path": "wikipedia"}


def llm_node(state: AgentState):
    response = llm.invoke(state["input"])
    return {"output": response.content, "path": "llm"}


# ---------------------------------------------------
# 6️⃣ Build LangGraph
# ---------------------------------------------------

def create_agent():

    graph = StateGraph(AgentState)

    graph.add_node("calculator_node", calculator_node)
    graph.add_node("wikipedia_node", wikipedia_node)
    graph.add_node("llm_node", llm_node)

    graph.set_conditional_entry_point(
        router,
        {
            "calculator_node": "calculator_node",
            "wikipedia_node": "wikipedia_node",
            "llm_node": "llm_node"
        }
    )

    graph.add_edge("calculator_node", END)
    graph.add_edge("wikipedia_node", END)
    graph.add_edge("llm_node", END)
    return graph.compile()


# ---------------------------------------------------
# 7️⃣ Example Execution
# ---------------------------------------------------

if __name__ == "__main__":

    agent = create_agent()

    queries = [
        "Calculate 25 * 18",
        "Who is the CEO of Tesla?",
        "Explain what Agentic AI means"
    ]

    for q in queries:
        result = agent.invoke({"input": q})
        print("\nQuery:", q)
        print("Path:", result["path"])
        print("Answer:", result["output"])


Query: Calculate 25 * 18
Path: calculator
Answer: Error: invalid syntax (<string>, line 1)

Query: Who is the CEO of Tesla?
Path: wikipedia
Answer: Page: Tesla Cybertruck
Summary: The Tesla Cybertruck is a battery-electric full-size pickup truck manufactured by Tesla, Inc. since 2023. It was unveiled as a prototype in November 2019, featuring a distinctive angular design composed of flat, unpainted stainless steel body panels, drawing comparisons to low-polygon computer models.
While scheduled for production in late 2021, the vehicle faced multiple delays before entering limited production at Gigafactory Texas in November 2023, with customer deliveries beginning later that month. As of 2026, two variants are available: a tri-motor all-wheel drive (AWD) model marketed as the "Cyberbeast" and a dual-motor AWD model. EPA range estimates vary by configuration, from 320 to 350 miles (515 to 565 km). 
As of 2026, the Cybertruck is sold in the United States, Mexico, Canada, South Korea, Qata

In [5]:
mermaid_diagram = agent.get_graph().draw_mermaid()
print(mermaid_diagram)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	calculator_node(calculator_node)
	wikipedia_node(wikipedia_node)
	llm_node(llm_node)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> calculator_node;
	__start__ -.-> llm_node;
	__start__ -.-> wikipedia_node;
	calculator_node --> __end__;
	llm_node --> __end__;
	wikipedia_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [4]:
# langgraph_agent_with_viz.py

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


# ============================================================
# 1️⃣ Define Agent State
# ============================================================

class AgentState(TypedDict):
    input: str
    output: str
    path: str


# ============================================================
# 2️⃣ Define Tools
# ============================================================

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"


wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())


# ============================================================
# 3️⃣ Initialize LLM
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


# ============================================================
# 4️⃣ Routing Logic
# ============================================================

def router(state: AgentState):
    query = state["input"].lower()

    if any(op in query for op in ["+", "-", "*", "/"]) or "calculate" in query:
        return "calculator_node"

    elif "who" in query or "when" in query or "search" in query:
        return "wikipedia_node"

    else:
        return "llm_node"


# ============================================================
# 5️⃣ Node Implementations
# ============================================================

def calculator_node(state: AgentState):
    expression = state["input"].replace("calculate", "")
    result = calculator.invoke(expression)
    return {
        "output": result,
        "path": "calculator"
    }


def wikipedia_node(state: AgentState):
    result = wiki.run(state["input"])
    return {
        "output": result,
        "path": "wikipedia"
    }


def llm_node(state: AgentState):
    response = llm.invoke(state["input"])
    return {
        "output": response.content,
        "path": "llm"
    }


# ============================================================
# 6️⃣ Build LangGraph
# ============================================================

def build_graph():

    builder = StateGraph(AgentState)

    builder.add_node("calculator_node", calculator_node)
    builder.add_node("wikipedia_node", wikipedia_node)
    builder.add_node("llm_node", llm_node)

    builder.set_conditional_entry_point(
        router,
        {
            "calculator_node": "calculator_node",
            "wikipedia_node": "wikipedia_node",
            "llm_node": "llm_node"
        }
    )

    builder.add_edge("calculator_node", END)
    builder.add_edge("wikipedia_node", END)
    builder.add_edge("llm_node", END)

    return builder


# ============================================================
# 7️⃣ Run + Visualization
# ============================================================

# if __name__ == "__main__":
    

#     # Build graph
#     builder = build_graph()

#     # 🔥 Visualization (Static Architecture)
#     print("\n=== Mermaid Diagram ===")
#     print(builder.get_graph().draw_mermaid())

#     # Optional: Save PNG (requires graphviz installed)
#     try:
#         builder.get_graph().draw_png("agent_graph.png")
#         print("Graph PNG saved as agent_graph.png")
#     except Exception:
#         print("PNG export skipped (graphviz may not be installed)")

#     # Compile agent
#     agent = builder.compile()

#     # Run example queries
#     queries = [
#         "Calculate 25 * 18",
#         "Who is the CEO of Tesla?",
#         "Explain Agentic AI"
#     ]

#     for q in queries:
#         result = agent.invoke({"input": q})

#         print("\nQuery:", q)
#         print("Path Selected:", result["path"])
#         print("Answer:", result["output"][:200], "...")




In [5]:
if __name__ == "__main__":

    # --------------------------------------------------
    # 1️⃣ Build Graph (Builder Phase)
    # --------------------------------------------------
    builder = build_graph()

    # --------------------------------------------------
    # 2️⃣ Compile Graph (Executable Phase)
    # --------------------------------------------------
    agent = builder.compile()

    # --------------------------------------------------
    # 3️⃣ Visualize Architecture (Static View)
    # --------------------------------------------------
    print("\n=== Mermaid Diagram ===")
    try:
        mermaid_diagram = agent.get_graph().draw_mermaid()
        print(mermaid_diagram)
    except Exception as e:
        print("Mermaid visualization failed:", str(e))

    # Optional: Save PNG (requires graphviz installed)
    try:
        agent.get_graph().draw_png("agent_graph.png")
        print("Graph PNG saved as agent_graph.png")
    except Exception:
        print("PNG export skipped (graphviz may not be installed)")

    # --------------------------------------------------
    # 4️⃣ Execute Example Queries
    # --------------------------------------------------
    queries = [
        "Calculate 25 * 18",
        "Who is the CEO of Tesla?",
        "Explain Agentic AI"
    ]

    for q in queries:
        result = agent.invoke({"input": q})

        print("\n--------------------------------")
        print("Query:", q)
        print("Path Selected:", result["path"])
        print("Answer:", result["output"][:200], "...")


=== Mermaid Diagram ===
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	calculator_node(calculator_node)
	wikipedia_node(wikipedia_node)
	llm_node(llm_node)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> calculator_node;
	__start__ -.-> llm_node;
	__start__ -.-> wikipedia_node;
	calculator_node --> __end__;
	llm_node --> __end__;
	wikipedia_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

PNG export skipped (graphviz may not be installed)

--------------------------------
Query: Calculate 25 * 18
Path Selected: calculator
Answer: Error: invalid syntax (<string>, line 1) ...

--------------------------------
Query: Who is the CEO of Tesla?
Path Selected: wikipedia
Answer: Page: List of predictions for autonomous Tesla vehicles by Elon Musk
Summary: This is a list of predictions for autonomous Tesla vehicles made by Elon Musk, CEO of Tesla, Inc. The predictions